# SCRFD Competition Offline Runner

Notebook template for Kaggle competition environments with internet disabled.

Expected Kaggle Dataset inputs:
- `scrfd-fullsearch-kaggle-src`: contains `scrfd-fullsearch-kaggle-offline.zip`
- `scrfd-wheelhouse`: optional wheel bundle if the competition image is missing packages
- `widerface-retinaface-format`: your WIDER FACE dataset in SCRFD format

In [ ]:
REPO_ZIP = ""
DATA_ROOT = ""
WHEELHOUSE = "/kaggle/input/scrfd-wheelhouse"

RUN_MODE = "baseline_train_quick"
# Options: baseline_train_quick, baseline_train, baseline_eval, step1_generate, step1_train, step1_eval,
# step1_stat, step2_generate, step2_train, step2_eval, step2_stat, full_search

GPU_ID = 0
TOTAL_EPOCHS = 16
EVAL_INTERVAL = 4
CHECKPOINT_INTERVAL = 4
IDX_FROM = 1
IDX_TO = 64
TEMPLATE = 10
CHECKPOINT = ""
THR = "0.02"
MODE_VALUE = "0"


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

def find_repo_source(explicit_path: str) -> Path:
    if explicit_path:
        candidate = Path(explicit_path)
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    zip_names = ['scrfd-fullsearch-kaggle-offline.zip', 'scrfd-fullsearch-kaggle-kagglefix-v2.zip']
    for zip_name in zip_names:
        matches = list(input_root.rglob(zip_name))
        if matches:
            return matches[0]
    for candidate in input_root.rglob('scrfd-fullsearch-kaggle'):
        if (candidate / 'scripts' / 'kaggle_competition_entry.py').exists():
            return candidate
    raise FileNotFoundError('Could not find SCRFD source dataset under /kaggle/input')

repo_root = Path('/kaggle/working/scrfd-fullsearch-kaggle')
if repo_root.exists():
    shutil.rmtree(repo_root)

repo_source = find_repo_source(REPO_ZIP)
print('Using repo source:', repo_source)
if repo_source.is_dir():
    source_root = repo_source / 'scrfd-fullsearch-kaggle'
    if source_root.exists():
        shutil.copytree(source_root, repo_root)
    else:
        shutil.copytree(repo_source, repo_root)
else:
    with zipfile.ZipFile(repo_source) as zf:
        zf.extractall('/kaggle/working')

assert repo_root.exists(), f'Missing repo after extraction: {repo_root}'
for shell_path in (repo_root / 'scripts').glob('*.sh'):
    shell_path.chmod(0o755)
for shell_path in (repo_root / 'tools').glob('*.sh'):
    shell_path.chmod(0o755)
for shell_path in (repo_root / 'search_tools').glob('*.sh'):
    shell_path.chmod(0o755)
os.chdir(repo_root)
print('Repo ready at', repo_root)


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def _is_retinaface_root(candidate: Path) -> bool:
    return (candidate / 'train' / 'labelv2.txt').exists() and (candidate / 'val' / 'labelv2.txt').exists()


def materialize_data_root(explicit_path: str) -> Path:
    input_root = Path('/kaggle/input')
    work_data_root = Path('/kaggle/working/retinaface_dataset')

    def from_candidate(candidate: Path) -> Path | None:
        if not candidate.exists():
            return None
        if candidate.is_dir():
            if _is_retinaface_root(candidate):
                return candidate
            nested = candidate / 'retinaface'
            if _is_retinaface_root(nested):
                return nested
            return None
        if candidate.suffix.lower() == '.zip':
            if work_data_root.exists():
                shutil.rmtree(work_data_root)
            work_data_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(candidate) as zf:
                zf.extractall(work_data_root)
            extracted = work_data_root / 'retinaface'
            if _is_retinaface_root(extracted):
                return extracted
        return None

    if explicit_path:
        resolved = from_candidate(Path(explicit_path))
        if resolved is not None:
            return resolved

    for candidate in input_root.rglob('retinaface'):
        resolved = from_candidate(candidate)
        if resolved is not None:
            return resolved
    for candidate in input_root.rglob('retinaface-kaggle-upload-with-test.zip'):
        resolved = from_candidate(candidate)
        if resolved is not None:
            return resolved
    for candidate in input_root.rglob('*.zip'):
        if 'retinaface' in candidate.name.lower() or 'widerface' in candidate.name.lower():
            resolved = from_candidate(candidate)
            if resolved is not None:
                return resolved
    raise FileNotFoundError('Could not find retinaface dataset root or retinaface dataset zip under /kaggle/input')


def materialize_wheelhouse(explicit_path: str) -> str | None:
    input_root = Path('/kaggle/input')
    work_wheelhouse = Path('/kaggle/working/scrfd_wheelhouse')

    def from_candidate(candidate: Path) -> str | None:
        if not candidate.exists():
            return None
        if candidate.is_dir():
            whl_files = list(candidate.glob('*.whl'))
            if whl_files:
                return str(candidate)
            nested = candidate / 'wheelhouse'
            if nested.exists() and list(nested.glob('*.whl')):
                return str(nested)
            return None
        if candidate.suffix.lower() == '.zip':
            if work_wheelhouse.exists():
                shutil.rmtree(work_wheelhouse)
            work_wheelhouse.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(candidate) as zf:
                zf.extractall(work_wheelhouse)
            for folder in [work_wheelhouse, work_wheelhouse / 'wheelhouse']:
                if folder.exists() and list(folder.glob('*.whl')):
                    return str(folder)
        return None

    if explicit_path:
        resolved = from_candidate(Path(explicit_path))
        if resolved is not None:
            return resolved

    for candidate in input_root.rglob('scrfd-wheelhouse.zip'):
        resolved = from_candidate(candidate)
        if resolved is not None:
            return resolved
    for candidate in input_root.rglob('*.whl'):
        return str(candidate.parent)
    return None


repo_root = Path('/kaggle/working/scrfd-fullsearch-kaggle')
entry = repo_root / 'scripts' / 'kaggle_competition_entry.py'
work_root = repo_root / 'work_dirs'
result_root = repo_root / 'wouts'
data_root = materialize_data_root(DATA_ROOT)
wheelhouse_root = materialize_wheelhouse(WHEELHOUSE)
print('Using data root:', data_root)
print('Using wheelhouse:', wheelhouse_root or 'none')

cmd = [
    sys.executable,
    str(entry),
    '--mode', RUN_MODE,
    '--data-root', str(data_root),
    '--work-root', str(work_root),
    '--result-root', str(result_root),
    '--gpu-id', str(GPU_ID),
    '--idx-from', str(IDX_FROM),
    '--idx-to', str(IDX_TO),
    '--template', str(TEMPLATE),
    '--thr', str(THR),
    '--mode-value', str(MODE_VALUE),
]

if TOTAL_EPOCHS is not None:
    cmd.extend(['--total-epochs', str(TOTAL_EPOCHS)])
if EVAL_INTERVAL is not None:
    cmd.extend(['--eval-interval', str(EVAL_INTERVAL)])
if CHECKPOINT_INTERVAL is not None:
    cmd.extend(['--checkpoint-interval', str(CHECKPOINT_INTERVAL)])
if CHECKPOINT:
    cmd.extend(['--checkpoint', CHECKPOINT])
if wheelhouse_root:
    cmd.extend(['--wheelhouse', wheelhouse_root])
else:
    cmd.append('--skip-offline-install')

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
